# Mastodon: one community at a time

Most Mastodon servers expose a public-timeline endpoint without a key (some large
instances, including mastodon.social, now require authentication; the code handles that). Because each server *is* a community (mastodon.social, hachyderm.io,
fosstodon.org), the instance you choose is a corpus choice — name it in your Data
Biography.

In [ ]:
import os
import pandas as pd
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"

RECORDED = [
 {"content": "<p>morning birdsong recording from the balcony</p>", "created_at": "2026-05-01T07:11:00Z", "language": "en"},
 {"content": "<p>new blog post: packaging python in 2026</p>", "created_at": "2026-05-01T07:12:30Z", "language": "en"},
]

def public_timeline(instance="fosstodon.org", limit=40):
    if SMOKE:
        return pd.DataFrame(RECORDED)
    try:
        import requests
        r = requests.get(f"https://{instance}/api/v1/timelines/public",
                         params={"limit": min(limit, 40), "local": "true"}, timeout=15)
        if r.status_code == 422:
            print(f"{instance} requires authentication for its public timeline "
                  "(large instances increasingly do); try another instance, e.g. "
                  "fosstodon.org, hachyderm.io, mastodon.art")
            return pd.DataFrame(RECORDED)
        r.raise_for_status()
        return pd.DataFrame([{ "content": p["content"], "created_at": p["created_at"],
                               "language": p.get("language")} for p in r.json()])
    except Exception as e:
        print("no network, using the recorded sample:", e)
        return pd.DataFrame(RECORDED)

posts = public_timeline()
print(len(posts), "posts from the public timeline")

In [ ]:
# Strip the HTML before counting; posts arrive as markup.
from bs4 import BeautifulSoup
posts["text"] = posts["content"].map(lambda h: BeautifulSoup(h, "html.parser").get_text(" "))
posts[["created_at", "text"]].head()